# 🏦 Lesson 07b — Multimodal Vision LLMs
**Domain:** Banking CRM · **Dataset:** Synthetic bank statements
**Duration:** ~2 hours

In [ ]:
# ── Imports & Setup ──────────────────────────────────────────────────────────
import os, json, time, base64, datetime
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # loads OPENAI_API_KEY and ANTHROPIC_API_KEY from .env

# ─ LLM clients ───────────────────────────────────────────────────
from openai import OpenAI
from anthropic import Anthropic

openai_client  = OpenAI()
claude_client  = Anthropic()

# ─ LM Studio (free local inference) ──────────────────────────────
lm_client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

# ─ Course checker ────────────────────────────────────────────────
from llm_checker import check, hint, evaluate, progress, show_exercise

print("✅  All imports OK — Module 7 ready")

---
## Lesson 07b — Multimodal Vision LLMs (~2 h)
### Processing Bank Documents with Vision

> ⚠️ **GDPR / Banking Compliance**  
> Sending real bank statements to GPT-4o (OpenAI cloud) **may violate GDPR and banking regulations**.  
> This lesson uses **ONLY synthetic statements** generated by code — no real financial data.  
> In production: consider on-premise Azure OpenAI, local vision models (LLaVA, Qwen-VL), or anonymise before cloud.

**Learning objectives:**
- Generate synthetic PDF/PNG bank statements with ReportLab
- Encode images to base64 for vision API input
- Extract structured Pydantic objects from document images
- Understand multimodal RAG patterns


In [ ]:
# ── Synthetic bank statement generator ───────────────────────────
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib.units import mm
from PIL import Image
import random, io

def generate_statement(output_path: str = "synthetic_statement.pdf", n_txn: int = 10) -> dict:
    """Generate a synthetic bank statement PDF. Returns ground-truth data dict."""
    random.seed(42)
    fake_names  = ["Anna Kowalska", "Piotr Nowak", "Marta Wiśniewska"]
    fake_ibans  = ["PL61 1090 1014 0000 0712 1981 2874",
                   "PL83 1140 2004 0000 3302 7894 1234"]
    name    = random.choice(fake_names)
    iban    = random.choice(fake_ibans)
    account_last4 = iban.replace(" ", "")[-4:]
    period_start  = "2024-01-01"
    period_end    = "2024-01-31"

    categories = ["Zakupy", "Przelew", "Wynagrodzenie", "Czynsz", "Internet", "Restauracja"]
    transactions = []
    for i in range(n_txn):
        is_credit = random.random() > 0.4
        amount    = round(random.uniform(10, 3000), 2)
        transactions.append({
            "date":    f"2024-01-{(i+1):02d}",
            "desc":    f"{random.choice(categories)} #{i+1}",
            "amount":  amount if is_credit else -amount,
            "type":    "credit" if is_credit else "debit",
        })

    # ─ Draw PDF ──────────────────────────────────────────────────
    c = canvas.Canvas(output_path, pagesize=A4)
    w, h = A4
    c.setFont("Helvetica-Bold", 16)
    c.drawString(20*mm, h - 25*mm, "SYNTHETIC BANK SA — Wyciąg z konta")
    c.setFont("Helvetica", 10)
    c.drawString(20*mm, h - 35*mm, f"Klient: {name}")
    c.drawString(20*mm, h - 42*mm, f"IBAN: {iban}")
    c.drawString(20*mm, h - 49*mm, f"Okres: {period_start} – {period_end}")
    c.setFont("Helvetica-Bold", 9)
    y = h - 62*mm
    c.drawString(20*mm, y, "Data"); c.drawString(50*mm, y, "Opis"); c.drawString(140*mm, y, "Kwota (PLN)")
    c.line(20*mm, y-2*mm, 190*mm, y-2*mm)
    c.setFont("Helvetica", 9)
    for txn in transactions:
        y -= 8*mm
        c.drawString(20*mm, y, txn["date"])
        c.drawString(50*mm, y, txn["desc"][:28])
        c.drawRightString(190*mm, y, f"{txn['amount']:+.2f}")
    total_credits = sum(t["amount"] for t in transactions if t["type"] == "credit")
    total_debits  = abs(sum(t["amount"] for t in transactions if t["type"] == "debit"))
    y -= 12*mm
    c.setFont("Helvetica-Bold", 10)
    c.drawString(20*mm, y, f"Suma wpływów: {total_credits:,.2f} PLN")
    c.drawString(110*mm, y, f"Suma wydatków: {total_debits:,.2f} PLN")
    c.save()

    return {
        "account_last4": account_last4,
        "period_start":  period_start,
        "period_end":    period_end,
        "transactions":  transactions,
        "total_credits": round(total_credits, 2),
        "total_debits":  round(total_debits, 2),
    }

ground_truth = generate_statement("synthetic_statement.pdf")
print("✅ Statement generated. Ground truth:")
print(json.dumps(ground_truth, indent=2, ensure_ascii=False))

In [ ]:
# ── Convert PDF page to PNG for vision API ────────────────────────
import subprocess

def pdf_to_png(pdf_path: str, png_path: str = "statement.png", dpi: int = 150):
    """Convert first page of PDF to PNG using pdftoppm (poppler)."""
    try:
        subprocess.run(
            ["pdftoppm", "-png", "-r", str(dpi), "-l", "1", "-f", "1",
             pdf_path, png_path.replace(".png", "")],
            check=True, capture_output=True
        )
        # pdftoppm appends -1 to filename
        import shutil
        shutil.move(png_path.replace(".png", "") + "-1.png", png_path)
        print(f"✅ PNG saved: {png_path}")
    except FileNotFoundError:
        # Fallback: use PIL/ReportLab to render (won't work without poppler)
        print("⚠️  pdftoppm not found — install poppler-utils: sudo apt install poppler-utils")
        print("   Alternatively, use reportlab to render directly to PNG.")

pdf_to_png("synthetic_statement.pdf")

In [ ]:
# ── Vision extraction with Pydantic ──────────────────────────────
from pydantic import BaseModel, Field
from typing import Optional, List

class Transaction(BaseModel):
    date:   str
    desc:   str
    amount: float

class BankStatement(BaseModel):
    account_last4:  str  = Field(description="Last 4 digits of account number")
    period_start:   str  = Field(description="Statement start date YYYY-MM-DD")
    period_end:     str  = Field(description="Statement end date YYYY-MM-DD")
    transactions:   List[Transaction]
    total_credits:  float
    total_debits:   float

def extract_statement(image_path: str) -> BankStatement:
    """Send synthetic statement image to GPT-4o vision and extract structured data."""
    with open(image_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode()

    response = openai_client.chat.completions.create(
        model="gpt-4o",
        max_tokens=1500,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{img_b64}"}
                },
                {
                    "type": "text",
                    "text": (
                        "Extract ALL data from this bank statement. "
                        "Reply with valid JSON matching this schema:\n"
                        "{ account_last4, period_start, period_end, "
                        "transactions: [{date, desc, amount}], "
                        "total_credits, total_debits }"
                    )
                }
            ]
        }]
    )
    raw = response.choices[0].message.content
    # Strip markdown fences if present
    raw = raw.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
    return BankStatement(**json.loads(raw))

extracted = extract_statement("statement.png")
print("Extracted:", extracted.model_dump_json(indent=2))

---
### 🟡 EXERCISE Ex 07b-1 — Synthetic Statement Generation + Extraction

1. Generate a synthetic bank statement (10 transactions, fake names, fake IBAN)
2. Convert to PNG and send to GPT-4o vision
3. Extract structured `BankStatement` Pydantic object
4. Verify extraction accuracy against ground truth

**Hints:**
- `from reportlab.lib.pagesizes import A4; from reportlab.pdfgen import canvas`
- `base64_img = base64.b64encode(open('statement.png', 'rb').read()).decode()`

**Auto-check verifies:**
- ✓ ≥ 8 transactions extracted from a 10-transaction statement
- ✓ `total_credits` within 5% of actual sum
- ✓ `account_last4` contains only digits (4 chars)


In [ ]:
show_exercise(
    "07b-1", "Synthetic Statement Generation + Extraction",
    "Generate a 10-transaction synthetic bank statement, convert to image, "
    "extract structured data with GPT-4o vision, validate against ground truth.",
    hints=[
        "▸ from reportlab.lib.pagesizes import A4; from reportlab.pdfgen import canvas",
        "▸ base64_img = base64.b64encode(open('statement.png', 'rb').read()).decode()",
        "▸ Use the BankStatement Pydantic model for structured extraction",
    ],
    checks=[
        "≥ 8 transactions extracted from 10-transaction statement",
        "total_credits within 5% of actual sum",
        "account_last4 is 4 digits"
    ]
)

In [ ]:
# ── YOUR SOLUTION — Ex 07b-1 ─────────────────────────────────────

# Generate your own statement (feel free to customize)
my_ground_truth = generate_statement("my_statement.pdf", n_txn=10)
pdf_to_png("my_statement.pdf", "my_statement.png")

# Extract with vision
my_extracted = None

# YOUR CODE HERE — call extract_statement("my_statement.png")
# ─────────────────────────────────────────────────────────────────

print("Ground truth transactions:", len(my_ground_truth["transactions"]))
print("Extracted transactions:", len(my_extracted.transactions) if my_extracted else "N/A")

In [ ]:
# ── Auto-check Ex 07b-1 ───────────────────────────────────────────
gt = my_ground_truth
ex = my_extracted

def within_5pct(a, b):
    return abs(a - b) / max(abs(b), 0.01) <= 0.05

check([
    (ex is not None,                          "Extraction returned a result"),
    (len(ex.transactions) >= 8,               "≥ 8 transactions extracted"),
    (within_5pct(ex.total_credits, gt["total_credits"]), "total_credits within 5% of actual"),
    (ex.account_last4.isdigit() and len(ex.account_last4) == 4, "account_last4 is 4 digits"),
    (ex.period_start == gt["period_start"],   "period_start matches"),
], exercise_id="07b-1")

---
## ✅ Lesson 07b Readiness Checklist
- ☐ Synthetic bank statement generated with ReportLab or HTML→PNG
- ☐ Vision extraction pipeline works with Pydantic structured output
- ☐ No real financial data used in any exercise